# 01 Preprocessing and Exploration

## Goal
Prepare a clean table for one focused question: **among reported wildlife strikes with a known damage label, can we classify whether aircraft damage was reported?**

The FAA defines `DAMAGE` as N (none), M (minor), M? (damaged but extent undetermined), S (substantial), and D (destroyed); blank means unknown. We use N as 0 and M/M?/S/D as 1. Unknown labels are excluded only from supervised modeling, not silently changed to no damage.

We select a small set of fields that describe the reported strike: incident month, time of day, phase of flight, aircraft class/mass, wildlife size/species, number struck, and height. Fields recorded after the outcome—damage details, effect on flight, repair cost, downtime, injuries, fatalities, and free-text remarks—must not be predictors.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

PROJECT_DIR = Path.cwd()
INPUT = PROJECT_DIR / "data" / "interim" / "strike_reports_standardized.csv"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
FIGURE_DIR = PROJECT_DIR / "outputs" / "figures"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT.exists():
    raise FileNotFoundError("Run 00_DATASET_SETUP_AND_PROFILE.ipynb first.")
df = pd.read_csv(INPUT, low_memory=False)
source_field_count = df.shape[1]
print("Loaded:", df.shape)

In [ ]:
# Remove duplicate report IDs only when the source supplies that identifier.
if "INDEX_NR" in df.columns:
    before = len(df)
    df = df.drop_duplicates(subset="INDEX_NR", keep="last")
    print("Duplicate report IDs removed:", before - len(df))
else:
    print("No INDEX_NR field; no automated duplicate removal performed.")

# Derive a year from the incident date if needed; keep date parsing explicit.
if "INCIDENT_DATE" in df.columns:
    parsed_date = pd.to_datetime(df["INCIDENT_DATE"], errors="coerce")
    if "INCIDENT_YEAR" not in df.columns:
        df["INCIDENT_YEAR"] = parsed_date.dt.year
else:
    parsed_date = pd.Series(pd.NaT, index=df.index)

if "INCIDENT_YEAR" not in df.columns or "DAMAGE" not in df.columns:
    raise KeyError("The selected source needs INCIDENT_YEAR (or INCIDENT_DATE) and DAMAGE.")

df["INCIDENT_YEAR"] = pd.to_numeric(df["INCIDENT_YEAR"], errors="coerce")
damage_code = df["DAMAGE"].astype("string").str.strip().str.upper()
damage_code = damage_code.str.extract(r"^(N|M[?]|M|S|D)(?:\s|$)", expand=False)
damage_map = {"N": 0, "M": 1, "M?": 1, "S": 1, "D": 1}
df["AIRCRAFT_DAMAGED"] = damage_code.map(damage_map).astype("Int64")

# The latest FAA annual report is through 2025. Exclude an incomplete newer year
# if the live export already contains it; record the resulting years in the output.
df = df.loc[df["INCIDENT_YEAR"].between(1990, 2025)].copy()
label_counts = df["AIRCRAFT_DAMAGED"].value_counts(dropna=False).rename("records").to_frame()
display(label_counts)
print("Rows with unknown damage outcome:", int(df["AIRCRAFT_DAMAGED"].isna().sum()))

In [ ]:
# A small, documented feature set: 7 categorical + 2 numeric source fields.
categorical_features = [
    "INCIDENT_MONTH", "TIME_OF_DAY", "PHASE_OF_FLT", "AC_CLASS",
    "AC_MASS", "SIZE", "SPECIES",
]
numeric_source_features = ["BIRDS_STRUCK", "HEIGHT"]
selected_features = categorical_features + numeric_source_features
missing_columns = [c for c in selected_features if c not in df.columns]
if missing_columns:
    raise KeyError(f"Expected FAA fields are missing: {missing_columns}. Check the selected sheet/column names.")

model_df = df[selected_features + ["INCIDENT_YEAR", "AIRCRAFT_DAMAGED"]].copy()
for col in numeric_source_features:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")
    # Impossible negative values become missing; extreme valid values are retained.
    model_df.loc[model_df[col] < 0, col] = np.nan
    # Long right tails are compressed without removing large real reports.
    model_df[col + "_LOG1P"] = np.log1p(model_df[col])

numeric_features = [c + "_LOG1P" for c in numeric_source_features]
feature_columns = categorical_features + numeric_features
model_df = model_df.dropna(subset=["AIRCRAFT_DAMAGED", "INCIDENT_YEAR"]).copy()
model_df["AIRCRAFT_DAMAGED"] = model_df["AIRCRAFT_DAMAGED"].astype(int)

for col in categorical_features:
    model_df[col] = model_df[col].astype("string").fillna("Unknown").str.strip()
    model_df.loc[model_df[col].eq(""), col] = "Unknown"

print("Numeric fields before/after log1p transformation:")
display(model_df[numeric_source_features + numeric_features].describe(percentiles=[.5,.9,.95,.99]).transpose())

print("Source columns before domain-based feature selection:", df.shape[1])
print("Selected predictor fields before encoding:", len(feature_columns))
print("Selected fields:", feature_columns)
print("Model table shape:", model_df.shape)
display(model_df[feature_columns + ["AIRCRAFT_DAMAGED"]].head())

## Exploratory graphs
These charts describe reports. They are not proof that a location, species, or season causes a strike. Use report counts as counts of reports, not as per-flight risk; this project does not merge flight-volume data.

In [ ]:
sns.set_theme(style="whitegrid")

year_counts = df.groupby("INCIDENT_YEAR").size().rename("reported_strikes")
ax = year_counts.plot(figsize=(10, 4), color="#24758a", linewidth=2)
ax.set(title="FAA wildlife strike reports by incident year", xlabel="Year", ylabel="Reported incidents")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "reports_by_year.png", dpi=150)
plt.show()

phase_rate = (model_df.groupby("PHASE_OF_FLT", dropna=False)["AIRCRAFT_DAMAGED"]
              .agg(reports="size", damage_share="mean")
              .query("reports >= 50")
              .sort_values("damage_share", ascending=False))
display(phase_rate)
ax = (phase_rate["damage_share"] * 100).sort_values().plot.barh(figsize=(8, 5), color="#e18a4b")
ax.set(title="Reported damage share by flight phase (groups with at least 50 reports)", xlabel="Reports with damage (%)", ylabel="Flight phase")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "damage_share_by_phase.png", dpi=150)
plt.show()

species_counts = model_df.groupby("SPECIES").agg(
    reports=("AIRCRAFT_DAMAGED", "size"),
    damage_share=("AIRCRAFT_DAMAGED", "mean"),
).query("reports >= 100").sort_values("reports", ascending=False).head(12)
display(species_counts)

In [ ]:
# Save the prepared modeling table and a compact preprocessing audit.
model_path = PROCESSED_DIR / "strike_model_table.csv"
model_df.to_csv(model_path, index=False)
audit = {
    "source_rows_after_deduplication_and_year_filter": int(len(df)),
    "rows_with_known_target": int(len(model_df)),
    "rows_without_known_target_excluded_from_model": int(df["AIRCRAFT_DAMAGED"].isna().sum()),
    "source_field_count": int(source_field_count),
    "selected_predictor_count_before_encoding": int(len(feature_columns)),
    "selected_predictors": feature_columns,
    "numeric_transform": "log1p for non-negative birds struck and height; invalid negative values set missing",
    "categorical_missingness": "retained as Unknown category; encoding/imputation fitted in training pipeline",
    "damage_target": "N=0; M, M?, S, D=1; blank/other unknown excluded",
}
(PROJECT_DIR / "outputs" / "preprocessing_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
print("Saved:", model_path)
print("Saved preprocessing audit.")

## Preprocessing choices for the evaluation criteria
- **Missing values:** unknown categorical values are kept as `Unknown`; numeric missing values are median-imputed inside the training-only pipeline. Unknown target labels are excluded from modeling and counted.
- **Encoding:** all category fields are nominal for this model and use one-hot encoding. `AC_MASS` has an order in the FAA codebook, but one-hot encoding avoids assuming equal spacing between weight classes.
- **Scaling:** standard scaling is used for logistic regression numeric inputs; Random Forest does not need scaling.
- **Transformation:** `log1p` compresses the long tails of bird count and height. We do not remove valid high values.
- **Binning:** no extra binning is applied; month and flight phase already provide understandable groups.
- **Outliers:** investigate extreme values; only impossible negative count/height values are changed to missing. No arbitrary percentile trimming.
- **Feature selection/reduction:** documented domain-based selection reduces the source fields to nine predictors, before one-hot encoding. Post-event fields and identifiers are excluded to prevent leakage.
- **Leakage prevention:** imputers, encoders, scalers, and category grouping are fitted later using training data only.